# Dissipative quantum cooling

This notebook prepares approximate thermal states for the 4x4 transverse-field Ising model. Each cycle evolves the system with a fresh three-qubit bath, measures the bath, and keeps the conditional system state. After burn-in, the collected states are sampled in the X, Y, and Z bases for the challenge submission.

The depolarizing model is implemented as a stochastic Pauli unraveling. This is equivalent to `DepolarizingChannel(p)` after averaging trajectories, while keeping the 19-qubit simulation on a memory-efficient state-vector device.

## 1. Imports and fixed parameters

The scorer expects 16 system spins. The three bath qubits are reset after every interaction and therefore do not appear in the final submission.

In [12]:
import json
import sys
from pathlib import Path

import numpy as np
import pennylane as qml

rng = np.random.default_rng(0)

SYSTEM_QUBITS = 16
BATH_QUBITS = 3
COUPLING = 1.0
FIELD = 1.0
BATH_FIELD = max(2 * FIELD, 4 * COUPLING)
INTERACTION_STRENGTH = 0.25
TIME_STEP = np.pi / 40
RANDOMIZATION_SCALE = 1.0
TEMPERATURES = (0.01, 1.1875, 2.375, 4.0)
N_SHOTS = 128
N_TRAJECTORIES = 4
N_CYCLES = 20
BURN_IN = 10
SNAPSHOT_INTERVAL = 5

SYSTEM_WIRES = list(range(SYSTEM_QUBITS))
BATH_WIRES = list(range(SYSTEM_QUBITS, SYSTEM_QUBITS + BATH_QUBITS))
TOTAL_WIRES = SYSTEM_QUBITS + BATH_QUBITS
PERIODIC_EDGES = [(i, (i + 1) % SYSTEM_QUBITS) for i in SYSTEM_WIRES]
BATH_ZERO = np.eye(2**BATH_QUBITS, dtype=complex)[:, 0]

# p is the probability of a non-identity depolarizing error.
DEPOLARIZING_PROBABILITY = 0.002

## 2. Temperature-dependent interaction kernel

The Gaussian kernel controls the duration and strength of the dissipative interaction. It is renormalized so that the discrete sum remains comparable when the temperature changes.

In [13]:
def configure_temperature(temperature):
    """Return time indices and a normalized bath-coupling kernel for T."""
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    beta = 1.0 / temperature
    kernel_width = np.sqrt(4 * BATH_FIELD / beta)
    kernel_time = 3 / kernel_width
    n_steps = int(np.ceil(kernel_time / TIME_STEP))
    time_indices = np.arange(-n_steps, n_steps + 1)
    kernel = np.exp(-0.5 * (kernel_width * TIME_STEP * time_indices) ** 2)
    kernel /= TIME_STEP * np.abs(kernel).sum()
    return time_indices, kernel

## 3. System, bath, and interaction layers

Keeping these layers separate makes the Floquet order explicit: system evolution, bath evolution, then system-bath coupling.

In [14]:
def apply_system_layer():
    """Apply one periodic transverse-field Ising step."""
    for left, right in PERIODIC_EDGES:
        qml.PauliRot(-2 * TIME_STEP * COUPLING, "ZZ", wires=[left, right])
    for wire in SYSTEM_WIRES:
        qml.RX(-2 * TIME_STEP * FIELD, wires=wire)


def apply_bath_layer():
    """Rotate each bath qubit in the bath Hamiltonian."""
    for wire in BATH_WIRES:
        qml.RX(-TIME_STEP * BATH_FIELD, wires=wire)


def apply_system_bath_layer(kernel_value, coupling_sites):
    """Couple each bath qubit to one selected system site."""
    angle = 2 * TIME_STEP * INTERACTION_STRENGTH * kernel_value / np.sqrt(2)
    for bath_wire, system_wire in zip(BATH_WIRES, coupling_sites):
        qml.PauliRot(angle, "ZY", wires=[system_wire, bath_wire])
        qml.PauliRot(angle, "YY", wires=[system_wire, bath_wire])

## 4. Depolarizing noise and joint evolution

For `DepolarizingChannel(p)`, sample I with probability `1 - 3p/4`, and X, Y, or Z with probability `p/4` each. Averaging these pure-state trajectories gives the same channel as a density-matrix simulation. Noise is applied after every logical layer to the qubits touched by that layer; set `DEPOLARIZING_PROBABILITY = 0.0` for a noiseless comparison.

In [15]:
def apply_depolarizing_noise(wires):
    """Sample the exact Pauli unraveling of a depolarizing channel."""
    p = DEPOLARIZING_PROBABILITY
    probabilities = (1 - 3 * p / 4, p / 4, p / 4, p / 4)
    for wire in wires:
        error = rng.choice(4, p=probabilities)
        if error == 1:
            qml.PauliX(wires=wire)
        elif error == 2:
            qml.PauliY(wires=wire)
        elif error == 3:
            qml.PauliZ(wires=wire)


def apply_noisy_system_layer():
    apply_system_layer()
    apply_depolarizing_noise(SYSTEM_WIRES)


def apply_noisy_bath_layer():
    apply_bath_layer()
    apply_depolarizing_noise(BATH_WIRES)


def apply_noisy_system_bath_layer(kernel_value, coupling_sites):
    apply_system_bath_layer(kernel_value, coupling_sites)
    apply_depolarizing_noise(list(coupling_sites) + BATH_WIRES)


evolution_device = qml.device("metal.qubit", wires=TOTAL_WIRES)


@qml.qnode(evolution_device)
def evolve_joint_state(system_state, coupling_sites, kernel, randomization_steps):
    """Prepare a fresh bath and return one noisy pure-state trajectory."""
    qml.StatePrep(np.kron(system_state, BATH_ZERO), wires=range(TOTAL_WIRES))
    for kernel_value in kernel:
        apply_noisy_system_layer()
        apply_noisy_bath_layer()
        apply_noisy_system_bath_layer(kernel_value, coupling_sites)
    for _ in range(randomization_steps):
        apply_noisy_system_layer()
    return qml.state()

## 5. Bath measurement and trajectory sampling

Reshaping separates system and bath indices. Sampling a bath basis state is the measurement-and-reset step; the selected column is the corresponding conditional system state.

In [16]:
def reset_bath(system_state, kernel):
    """Run one reset cycle and return the normalized system state."""
    coupling_sites = rng.choice(SYSTEM_QUBITS, size=BATH_QUBITS, replace=False)
    randomization_steps = int(
        rng.exponential(RANDOMIZATION_SCALE * len(kernel))
    ) if RANDOMIZATION_SCALE > 0 else 0
    joint_state = evolve_joint_state(
        system_state, coupling_sites, kernel, randomization_steps
    )
    joint_state = joint_state.reshape(2**SYSTEM_QUBITS, 2**BATH_QUBITS)
    bath_probabilities = np.linalg.norm(joint_state, axis=0) ** 2
    bath_probabilities /= bath_probabilities.sum()
    measured_bath = rng.choice(2**BATH_QUBITS, p=bath_probabilities)
    system_state = joint_state[:, measured_bath]
    return system_state / np.linalg.norm(system_state)


def run_trajectory(kernel, n_cycles=N_CYCLES, burn_in=BURN_IN,
                   snapshot_interval=SNAPSHOT_INTERVAL):
    """Collect system-state snapshots after the burn-in period."""
    system_state = np.zeros(2**SYSTEM_QUBITS, dtype=complex)
    system_state[rng.integers(2**SYSTEM_QUBITS)] = 1.0
    snapshots = []
    for cycle in range(n_cycles):
        system_state = reset_bath(system_state, kernel)
        if cycle >= burn_in and cycle % snapshot_interval == 0:
            snapshots.append(system_state.copy())
    return snapshots

## 6. Basis measurements and submission files

The scorer needs 128 binary outcomes for each basis. PennyLane returns eigenvalues ±1, so `(1 - eigenvalue) / 2` converts them to challenge bits.

In [17]:
submission_device = qml.device("metal.qubit", wires=SYSTEM_QUBITS)


@qml.qnode(submission_device, shots=1)
def sample_snapshot(snapshot, basis):
    qml.StatePrep(snapshot, wires=SYSTEM_WIRES)
    observable = {"X": qml.PauliX, "Y": qml.PauliY, "Z": qml.PauliZ}[basis]
    return tuple(qml.sample(observable(wire)) for wire in SYSTEM_WIRES)


def collect_submission(temperature):
    """Generate one scorer-compatible submission for a temperature."""
    _, kernel = configure_temperature(temperature)
    snapshot_pool = []
    for _ in range(N_TRAJECTORIES):
        snapshot_pool.extend(run_trajectory(kernel))
    if not snapshot_pool:
        raise RuntimeError("no snapshots were collected")

    chosen = rng.choice(len(snapshot_pool), size=N_SHOTS, replace=True)
    submission = {"temperature": temperature}
    for basis in ("X", "Y", "Z"):
        rows = []
        for snapshot_index in chosen:
            eigenvalues = np.asarray(
                sample_snapshot(snapshot_pool[snapshot_index], basis)
            ).reshape(-1)
            rows.append(((1 - eigenvalues) // 2).astype(int).tolist())
        submission[basis] = rows
    return submission


submissions = {}
submission_paths = {}
for temperature in TEMPERATURES:
    print(f"Collecting T={temperature}")
    submission = collect_submission(temperature)
    path = Path(f"submission_{temperature:.4g}.json")
    path.write_text(json.dumps(submission, indent=2) + "\n")
    submissions[temperature] = submission
    submission_paths[temperature] = path
    print(f"Wrote {path.resolve()}")

Wrote /Users/edgarroussel/projects/pq-hack/CIRAQLE/submission_0.01.json
Wrote /Users/edgarroussel/projects/pq-hack/CIRAQLE/submission_1.188.json
Wrote /Users/edgarroussel/projects/pq-hack/CIRAQLE/submission_2.375.json
Wrote /Users/edgarroussel/projects/pq-hack/CIRAQLE/submission_4.json


## 7. Optional local scoring

Scoring is separate from state preparation so the generated submissions can be inspected before evaluating them.

In [18]:
project_dir = Path.cwd()
if not (project_dir / "score.py").exists():
    project_dir = Path("/Users/edgarroussel/projects/pq-hack")
if not (project_dir / "score.py").exists():
    raise FileNotFoundError("Could not find score.py")

sys.path.insert(0, str(project_dir))
from score import score_submission

results = {
    temperature: score_submission(submission)
    for temperature, submission in submissions.items()
}
results_path = Path("submission_results.json")
results_path.write_text(json.dumps(results, indent=2) + "\n")
print(f"Results saved to: {results_path.resolve()}")
print(json.dumps(results, indent=2))

Results saved to: /Users/edgarroussel/projects/pq-hack/CIRAQLE/submission_results.json
{
  "0.01": {
    "temperature": 0.01,
    "score": 8.657334753723946,
    "single_site_score": 0.5797780166923042,
    "pair_correlation_score": 8.077556737031642,
    "estimated_magnetizations": {
      "X": -0.0009765625,
      "Y": 0.001953125,
      "Z": -0.0087890625
    },
    "exact_magnetizations": {
      "X": 0.25267421576725413,
      "Y": 0.0,
      "Z": 0.0
    }
  },
  "1.1875": {
    "temperature": 1.1875,
    "score": 8.226538429559223,
    "single_site_score": 0.5414713071125046,
    "pair_correlation_score": 7.6850671224467195,
    "estimated_magnetizations": {
      "X": 0.01171875,
      "Y": -0.015625,
      "Z": 0.0380859375
    },
    "exact_magnetizations": {
      "X": 0.2535219743536098,
      "Y": 0.0,
      "Z": 0.0
    }
  },
  "2.375": {
    "temperature": 2.375,
    "score": 4.160045199990426,
    "single_site_score": 0.844534550296934,
    "pair_correlation_score": 3.